In [0]:
from pyspark.sql import functions as F
CATALOG = "dbw_taxrisk_dev"
dbutils.widgets.text("country_code", "CO")
COUNTRY_CODE = dbutils.widgets.get("country_code")

In [0]:
rule = (spark.table(f"{CATALOG}.config.country_tax_rules")
        .filter(F.col("country_code") == COUNTRY_CODE)
        .orderBy(F.desc("valid_from")).first())
rate = float(rule["assumed_income_rate"])
print("regla:", rule["valid_from"], "| tasa:", rate, "| placeholder:", rule["is_placeholder"])

probs = {r["pred_class"]: float(r["p_in_kind"])
         for r in spark.table(f"{CATALOG}.config.class_calibration")
                       .filter(F.col("country_code") == COUNTRY_CODE).collect()}
pmap = F.create_map(*[x for k, v in probs.items() for x in (F.lit(k), F.lit(v))])

p = (spark.table(f"{CATALOG}.silver.posts_classified")
     .withColumn("p_in_kind", F.element_at(pmap, F.col("pred_class"))))

def exp_val(col):
    return F.coalesce(F.sum(F.col("p_in_kind") * F.col(col)), F.lit(0.0))

exposure = (p.groupBy("creator_key")
    .agg(exp_val("value_low").alias("expected_value_low"),
         exp_val("value_mid").alias("expected_value_mid"),
         exp_val("value_high").alias("expected_value_high"))
    .withColumn("tax_exposure_low",  F.col("expected_value_low")  * F.lit(rate))
    .withColumn("tax_exposure_mid",  F.col("expected_value_mid")  * F.lit(rate))
    .withColumn("tax_exposure_high", F.col("expected_value_high") * F.lit(rate))
    .withColumn("rate_used", F.lit(rate))
    .withColumn("rule_is_placeholder", F.lit(bool(rule["is_placeholder"])))
    .join(spark.table(f"{CATALOG}.silver.creators")
          .select("creator_key", "declared_income", "is_registered_taxpayer"), "creator_key", "left")
    .withColumn("value_to_declared_income",
                F.col("expected_value_mid") / F.nullif(F.col("declared_income"), F.lit(0))))

(exposure.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.gold.creator_exposure"))

In [0]:
g = spark.table(f"{CATALOG}.gold.creator_exposure")
tot = g.agg(F.sum("expected_value_low").alias("low"),
            F.sum("expected_value_mid").alias("mid"),
            F.sum("expected_value_high").alias("high")).first()
print(f"valor esperado total  bajo/medio/alto: {tot['low']:,.0f} / {tot['mid']:,.0f} / {tot['high']:,.0f}")
print("valor real de canjes (ground truth): 219,803,368")
display(g.orderBy(F.desc("tax_exposure_mid")).select(
    "creator_key", "expected_value_mid", "tax_exposure_mid",
    "declared_income", "value_to_declared_income", "is_registered_taxpayer").limit(10))